<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/Natural_Language_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Shakespeare dataset

In [1]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")

    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)

    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [2]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [3]:
vocab = sorted(set(shakespeare_text.lower()))
print("".join(vocab))


 !$&',-.3:;?abcdefghijklmnopqrstuvwxyz


In [4]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}

print(char_to_id["a"])
print(id_to_char[13])

13
a


In [5]:
import torch

def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [6]:
encoded = encode_text("Hello, world!")
print(encoded)



tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])


In [7]:
decoded = decode_text(encoded)
print(decoded)

hello, world!


In [8]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")

        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx + 1:end + 1]

        return window, target

In [9]:
window_length = 50
batch_size = 512

train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)

In [10]:
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

In [15]:
x_batch, y_batch = next(iter(train_loader))

print(x_batch.shape)
print(y_batch.shape)

torch.Size([512, 50])
torch.Size([512, 50])


#Embedding

In [24]:
import torch
import torch.nn as nn

torch.manual_seed(42)

embed = nn.Embedding(5, 3) # (vocab_size, embed_dim)

input_ids = torch.tensor([
    [3, 2],
    [0, 2]
])

output = embed(input_ids)

print(output)

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)


In [17]:
print(embed.weight)

Parameter containing:
tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617],
        [ 0.2674,  0.5349,  0.8094],
        [ 1.1103, -1.6898, -0.9890]], requires_grad=True)


In [28]:
vocab_size = len(vocab)
embed_dim = 16

embedding = nn.Embedding(vocab_size, embed_dim)

x_embed = embedding(X_batch)

print("vocab_size:", vocab_size)
print("x_batch shape:", X_batch.shape)
print("embedding.weight shape:", embedding.weight.shape)
print("x_embed shape:", x_embed.shape)

vocab_size: 39
x_batch shape: torch.Size([512, 50])
embedding.weight shape: torch.Size([39, 16])
x_embed shape: torch.Size([512, 50, 16])


#Building and Training the Char-RNN Model

In [29]:
import torch.nn.functional as F


In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [31]:
class ShakespeareModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=10,
        hidden_dim=128,
        dropout=0.1
    ):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, hidden_states = self.gru(embeddings)
        logits = self.output(outputs)

        return logits.permute(0, 2, 1) #[512, 50, 39] → [512, 39, 50] => [batch, vocab_size, sequence_length] Crossentropy[]

In [32]:
torch.manual_seed(42)

model = ShakespeareModel(vocab_size=len(vocab)).to(device)

In [33]:
x_batch, y_batch = next(iter(train_loader))

x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

logits = model(x_batch)

print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)
print("logits shape:", logits.shape)

x_batch shape: torch.Size([512, 50])
y_batch shape: torch.Size([512, 50])
logits shape: torch.Size([512, 39, 50])


In [34]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [35]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)


        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        total_correct += (predictions == y_batch).sum().item()
        total_tokens += y_batch.numel() # number of elements  if shape [2,3] then numel() 2*3= 6

    avg_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_tokens

    return avg_loss, accuracy

In [36]:
def evaluate(model, data_loader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            total_loss += loss.item()

            predictions = logits.argmax(dim=1)
            total_correct += (predictions == y_batch).sum().item()
            total_tokens += y_batch.numel()

    avg_loss = total_loss / len(data_loader)
    accuracy = total_correct / total_tokens

    return avg_loss, accuracy

In [37]:
n_epochs = 5

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        valid_loader,
        loss_fn,
        device
    )

    print(
        f"Epoch {epoch + 1}/{n_epochs} | "
        f"train loss: {train_loss:.4f}, train acc: {train_acc:.4f} | "
        f"valid loss: {valid_loss:.4f}, valid acc: {valid_acc:.4f}"
    )

Epoch 1/5 | train loss: 1.7120, train acc: 0.4871 | valid loss: 1.5529, valid acc: 0.5303
Epoch 2/5 | train loss: 1.4225, train acc: 0.5584 | valid loss: 1.5015, valid acc: 0.5437
Epoch 3/5 | train loss: 1.3826, train acc: 0.5682 | valid loss: 1.4861, valid acc: 0.5476
Epoch 4/5 | train loss: 1.3625, train acc: 0.5731 | valid loss: 1.4817, valid acc: 0.5485
Epoch 5/5 | train loss: 1.3500, train acc: 0.5763 | valid loss: 1.4813, valid acc: 0.5482


In [38]:
model.eval()

text = "To be or not to b"

encoded_text = encode_text(text).unsqueeze(dim=0).to(device) # [n,c] => [ 1,n c]

with torch.no_grad():
    y_logits = model(encoded_text)

predicted_char_id = y_logits[0, :, -1].argmax().item()
predicted_char = id_to_char[predicted_char_id]

print(predicted_char)

e


In [39]:
model.eval()

text = "The king is the season'd the state and things"

encoded_text = encode_text(text).unsqueeze(dim=0).to(device)

with torch.no_grad():
    y_logits = model(encoded_text)

predicted_char_id = y_logits[0, :, -1].argmax().item()
predicted_char = id_to_char[predicted_char_id]

print(predicted_char)

#Generating Fake Shakespearean Text

In [73]:
torch.manual_seed(42)

probs = torch.tensor([[0.5, 0.4, 0.1]])

samples = torch.multinomial(
    probs,
    replacement=True,
    num_samples=8
)

print(samples)

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])


In [40]:
def next_char(model, text, temperature=1.0):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)

    with torch.no_grad():
        y_logits = model(encoded_text)

    last_token_logits = y_logits[0, :, -1]

    y_probas = F.softmax(last_token_logits / temperature, dim=-1)

    predicted_char_id = torch.multinomial(
        y_probas,
        num_samples=1
    ).item()

    return id_to_char[predicted_char_id]

In [41]:
def extend_text(model, text, n_chars=80, temperature=1.0):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)

    return text

In [76]:
print(extend_text(
    model,
    "To be or not to b",
    n_chars=80,
    temperature=0.01
))

To be or not to be the state of the sea of the sea of the sea
when the season and the state of th


In [77]:
print(extend_text(
    model,
    "To be or not to b",
    n_chars=80,
    temperature=0.4
))

To be or not to be so death.

cominius:
i have more than he will seem false
to the should be the 


In [79]:
print(extend_text(
    model,
    "To be or not to b",
    n_chars=80,
    temperature=10
))

To be or not to b;-dd?f-vkinst$;.d;bdglon-;!eop;wh oechrino,tenv; o$;uqo
ryplssgck g'sustueb,
'zd
